
# Pipeline de datos — Clasificación de textos en Esperanto

**Asignatura:** Minería de Textos  
**Curso:** 2026–2027

## Objetivo

Este notebook desarrolla las dos primeras fases del proyecto de clasificación temática de textos en Esperanto:

1. **Construcción del dataset:** recopilación de párrafos mediante web scraping, organización de los textos y preparación de sus etiquetas.
2. **Procesamiento NLP:** aplicación de una librería propia de Esperanto para tokenización, lematización, etiquetado gramatical y gestión de stopwords.

Se han seleccionado tres categorías semánticamente relacionadas:

- **Ciberseguridad**
- **Inteligencia Artificial (IA)**
- **Programación**

El sistema utiliza un enfoque multietiqueta: un mismo párrafo puede pertenecer a varias categorías.

El resultado de este notebook es un dataset procesado que se utilizará posteriormente para entrenar los clasificadores en `pipeline_modelos.ipynb`.



In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

from esperanto_nlp.libreria import EsperantoNLP, preparar_texto
from esperanto_nlp.procesar_dataset import procesar_dataset

ETIQUETAS = ["Ciberseguridad", "IA", "Programacion"]

RUTA_ORIGINAL = Path("data/originales/dataset_multietiqueta.csv")
RUTA_PROCESADO = Path(
    "data/procesados/dataset_multietiqueta_procesado.csv"
)

print("Librerías importadas correctamente.")
print("Dataset original disponible:", RUTA_ORIGINAL.is_file())

: 


## 1. Construcción del dataset

### 1.1. Recopilación de textos

Los textos se recopilaron mediante web scraping de páginas en Esperanto relacionadas con las tres categorías seleccionadas.

El proceso de construcción del dataset está implementado en los módulos `extraccion_web.py` y `crear_dataset.py`.

Se recopilaron **1.659 párrafos procedentes de 120 artículos**, conservando información sobre su URL de origen.

Para facilitar la reproducción de los experimentos, el corpus recopilado se almacena en un CSV y no es necesario descargar nuevamente todas las páginas cada vez que se ejecuta el notebook.

### 1.2. Etiquetado multietiqueta

Inicialmente, los párrafos heredaban la categoría temática de su artículo.

Sin embargo, algunos textos abordan varias temáticas simultáneamente. Para representar este solapamiento se preparó una propuesta de etiquetado con tres columnas binarias independientes.

- `1`: el párrafo se considera perteneciente a la categoría.
- `0`: el párrafo no se considera perteneciente a la categoría.
- Valor vacío: anotación pendiente de revisión.

Las etiquetas son propuestas asistidas, no anotaciones completamente verificadas. Por ello, los registros pendientes se identifican y se excluyen del entrenamiento inicial.


In [ ]:
# Seleccionar una URL real del dataset
df_original = pd.read_csv(
    RUTA_ORIGINAL,
    sep=";",
    encoding="utf-8-sig",
    keep_default_na=False
)

url_ejemplo = df_original["url"].iloc[0]

print("URL de ejemplo:")
print(url_ejemplo)

try:
    respuesta = requests.get(
        url_ejemplo,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=15
    )
    respuesta.raise_for_status()

    soup = BeautifulSoup(respuesta.text, "html.parser")

    contenedor = soup.select_one(
        "div.itemBody.kl-blog-post-body"
    )

    if contenedor is None:
        contenedor = (
            soup.select_one("article")
            or soup.select_one("main")
        )

    if contenedor is None:
        raise ValueError("No se encontró el contenido principal.")

    parrafos_web = [
        p.get_text(" ", strip=True)
        for p in contenedor.select("p")
        if p.get_text(" ", strip=True)
    ]

    print("\nPárrafos extraídos:", len(parrafos_web))

    if parrafos_web:
        print("\nEjemplo de párrafo:")
        print(parrafos_web[0][:500])

except (requests.RequestException, ValueError) as error:
    print("\nNo se pudo extraer la página de ejemplo.")
    print("Motivo:", error)
    print("El dataset original guardado sigue disponible.")
    

: 


### 1.3. Análisis del corpus

Antes del procesamiento lingüístico, analizamos la composición del dataset.

Se contabilizan los artículos, los párrafos por categoría, los casos con varias etiquetas y las anotaciones pendientes de revisión.

Dado que las categorías no son excluyentes, la suma de sus frecuencias puede superar el número total de párrafos.

Este análisis permite comprobar la distribución del corpus y documentar posibles problemas de calidad de las etiquetas.


In [ ]:
# Estados de las anotaciones
print("Total de párrafos:", len(df_original))
print("Artículos diferentes:", df_original["url"].nunique())

print("\nEstados de revisión:")
print(df_original["estado_revision"].value_counts())

# Seleccionar propuestas completas
df_validos = df_original[
    df_original["estado_revision"] == "propuesto"
].copy()

for categoria in ETIQUETAS:
    df_validos[categoria] = df_validos[categoria].astype(int)

df_validos["numero_etiquetas"] = (
    df_validos[ETIQUETAS].sum(axis=1)
)

print("\nPárrafos con etiquetas propuestas:", len(df_validos))

print(
    "Párrafos con varias etiquetas:",
    (df_validos["numero_etiquetas"] > 1).sum()
)

frecuencias = df_validos[ETIQUETAS].sum()

print("\nDistribución por categoría:")
print(frecuencias)

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))

barras = ax.bar(
    frecuencias.index,
    frecuencias.values
)

ax.set_title("Distribución de categorías del dataset")
ax.set_xlabel("Categoría")
ax.set_ylabel("Número de párrafos")
ax.bar_label(barras, padding=3)
ax.set_ylim(0, frecuencias.max() * 1.15)

plt.tight_layout()
plt.show()

# Mostrar ejemplos con varias etiquetas
print("\nEjemplos multietiqueta:")

display(
    df_validos.loc[
        df_validos["numero_etiquetas"] > 1,
        ["text"] + ETIQUETAS
    ].head(5)
)

: 


## 2. Procesamiento del lenguaje natural

### 2.1. Librería EsperantoNLP

Se ha desarrollado una librería propia para procesar textos en Esperanto, aprovechando las reglas gramaticales regulares del idioma.

La implementación incluye:

- **Tokenización:** división de los textos en tokens.
- **Lematización:** obtención de las formas base de las palabras.
- **POS-tagging:** asignación de categorías gramaticales.
- **Análisis morfológico:** identificación de características como número, caso y tiempo verbal.
- **Stopwords:** identificación de palabras poco informativas.
- **Objetos Doc y Token:** representación estructurada del análisis lingüístico.

La librería está implementada en `libreria.py` mediante reglas propias, sin depender de modelos lingüísticos preentrenados.

A continuación, se demuestra su funcionamiento utilizando un párrafo real del dataset.


In [ ]:
# Inicializar nuestra librería
nlp = EsperantoNLP(conservar_negaciones=True)

# Seleccionar un ejemplo real
texto_ejemplo = df_validos.iloc[0]["text"]

# Obtener documento lingüístico
doc = nlp(texto_ejemplo)

# Construir tabla de tokens
datos_tokens = []

for token in list(doc)[:25]:
    datos_tokens.append({
        "Token": token.text,
        "Lema": token.lemma_,
        "POS": token.pos_,
        "Morfología": str(token.morph),
        "Stopword": token.is_stop
    })

df_tokens = pd.DataFrame(datos_tokens)

print("TEXTO ORIGINAL:")
print(texto_ejemplo)

print("\nANÁLISIS DE LOS PRIMEROS 25 TOKENS:")
display(df_tokens)

# Aplicar preprocesamiento para clasificación
tokens_procesados = preparar_texto(
    texto_ejemplo,
    nlp,
    quitar_numeros=True
)

print("\nTEXTO PROCESADO:")
print(" ".join(tokens_procesados))

print("\nPalabras originales:", len(texto_ejemplo.split()))
print("Tokens procesados:", len(tokens_procesados))

: 


### 2.2. Procesamiento del dataset completo

Una vez comprobado el funcionamiento de nuestra librería, aplicamos el preprocesamiento a los párrafos del corpus.

El procedimiento utiliza `EsperantoNLP` y `preparar_texto`, conservando los identificadores, las URLs y las etiquetas originales.

El resultado se guarda en `data/procesados/dataset_multietiqueta_procesado.csv`.

Se mantienen todos los registros para conservar la trazabilidad. Los textos que quedan vacíos tras el procesamiento y las anotaciones pendientes se identifican para que no sean utilizados indebidamente durante el entrenamiento.

La representación TF-IDF se realizará posteriormente en `pipeline_modelos.ipynb`, utilizando exclusivamente los datos de entrenamiento para ajustar el vocabulario.


In [ ]:
# Procesar el dataset completo mediante nuestra librería
procesar_dataset(
    entrada=RUTA_ORIGINAL,
    salida=RUTA_PROCESADO
)

print("\nDataset procesado disponible:", RUTA_PROCESADO.is_file())

In [ ]:
# Cargar el resultado del procesamiento
df_procesado = pd.read_csv(
    RUTA_PROCESADO,
    sep=";",
    encoding="utf-8-sig",
    keep_default_na=False,
    dtype={"fila_dataset": str}
)

# Comprobar la integridad de los datos
assert len(df_procesado) == len(df_original), (
    "Se han perdido registros durante el procesamiento."
)

assert (
    df_procesado["fila_dataset"].tolist()
    == df_original["fila_dataset"].astype(str).tolist()
), "Los identificadores no coinciden."

assert (
    df_procesado["url"].tolist()
    == df_original["url"].tolist()
), "Las URLs no coinciden."

for categoria in ETIQUETAS:
    assert (
        df_procesado[categoria].astype(str).tolist()
        == df_original[categoria].astype(str).tolist()
    ), f"Las etiquetas de {categoria} han cambiado."

# Comprobar los textos disponibles para el modelo
utilizables = df_procesado[
    (df_procesado["estado_revision"] == "propuesto")
    & (df_procesado["text"].str.strip() != "")
]

print("=" * 55)
print("PIPELINE DE DATOS COMPLETADO")
print("=" * 55)

print("Registros conservados:", len(df_procesado))
print("Textos utilizables:", len(utilizables))
print("Artículos diferentes:", df_procesado["url"].nunique())

print("\nArchivo generado:")
print(RUTA_PROCESADO.resolve())

print("\nEjemplos de textos procesados:")

display(
    df_procesado[
        ["fila_dataset", "text_original", "text"] + ETIQUETAS
    ].head(5)
)

print("\nDataset preparado para pipeline_modelos.ipynb")